# Stage 2 — Machine Learning (Regression + Classification)

This notebook is **Stage 2** of the split pipeline. It contains the original Regression (Sec. 12), Classification (Sec. 13), Train/Test split note (Sec. 15), the 5 post-cleaning/ML plots (Sec. 16), Model Results Summary (Sec. 18), and Final Conclusions (Sec. 19) — copied over **unchanged**.

It depends on Stage 1's output (`session_dataset.parquet`), loaded in the setup cell below instead of re-reading raw HDFS data.

**Requires:** `session_dataset.parquet` (from Stage 1)

In [ ]:
# --- Added for orchestration: scoped dependency install ---
# Revised from the original notebook's single install cell (which
# installed pyarrow + xgboost[spark] together for the whole notebook).
# This notebook only needs: pyarrow (fast Arrow transfer for .toPandas() plot calls) AND xgboost[spark] (SparkXGBRegressor / SparkXGBClassifier used in this notebook's regression and classification sections).
import subprocess
import sys

packages = [
    'pyarrow',
    'xgboost[spark]'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])
    print(pkg, 'ready')

print('All dependencies installed.')

In [ ]:
# --- Added for orchestration: Spark session + load Stage 1 output ---
import os
DATA_DIR = os.environ.get("ECOMM_DATA_DIR", "/tmp/ecommerce_pipeline/data")
os.makedirs(DATA_DIR, exist_ok=True)
print("Using DATA_DIR:", DATA_DIR)

import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 100
import numpy as np

spark = (
    SparkSession.builder
    .appName("ML-Models")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "10000")
spark.sparkContext.setLogLevel("WARN")

session_dataset = spark.read.parquet(f"{DATA_DIR}/session_dataset.parquet")
session_dataset = session_dataset.cache()
print("Loaded session_dataset:", session_dataset.count(), "rows")

---
## 12. Regression — Spark MLlib

**Target:** `session_total_spending`.

Because the vast majority of sessions never purchase (see EDA Plot 1 / Section 7), a spending target is
extremely zero-inflated across *all* sessions. To build a meaningful, well-posed regression problem, we
restrict the regression dataset to sessions that **did** purchase (`will_purchase == 1`) and predict *how
much* a converting session spends, based purely on its pre-purchase browsing behavior. This is a realistic
business question: "given a session that is converting, how much revenue will it generate?" 

In [ ]:
regression_data = session_dataset.filter(F.col("will_purchase") == 1)
print("Sessions with at least one purchase:", regression_data.count())

regression_data.select(
    "session_event_count", "session_view_count", "session_cart_count",
    "session_unique_products", "session_unique_brands", "session_unique_categories",
    "session_avg_price_viewed", "session_max_price_viewed", "session_duration_seconds",
    "session_start_hour", "session_dayofweek", "session_total_spending"
).summary("count", "mean", "stddev", "min", "max").show()

In [ ]:
from pyspark.ml.feature import VectorAssembler
try:
    from xgboost.spark import SparkXGBRegressor
except ImportError:
    from sparkxgb import XGBoostRegressor as SparkXGBRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

reg_feature_cols = [
    "session_event_count", "session_view_count", "session_cart_count",
    "session_unique_products", "session_unique_brands", "session_unique_categories",
    "session_avg_price_viewed", "session_max_price_viewed", "session_duration_seconds",
    "session_start_hour", "session_dayofweek",
]

regression_data = regression_data.na.drop(subset=reg_feature_cols + ["session_total_spending"])

reg_train, reg_test = regression_data.randomSplit([0.8, 0.2], seed=42)

# Cache training data: XGBoost iterates over it multiple times
# (one pass per boosting round). Without caching each round
# triggers a full re-read from HDFS.
reg_train = reg_train.cache()
clf_train_cached = False  # clf_train cached below
print("Regression train rows:", reg_train.count(), " | test rows:", reg_test.count())

reg_assembler = VectorAssembler(inputCols=reg_feature_cols, outputCol="features")

# Spark ML XGBoost Regressor (Spark MLlib Pipeline stage)
try:
    xgb_regressor = SparkXGBRegressor(
        features_col="features", label_col="session_total_spending",
        num_workers=2, n_estimators=100, max_depth=8, seed=42
    )
except Exception:
    xgb_regressor = SparkXGBRegressor(
        featuresCol="features", labelCol="session_total_spending"
    )

reg_pipeline = Pipeline(stages=[reg_assembler, xgb_regressor])
reg_model = reg_pipeline.fit(reg_train)

reg_predictions = reg_model.transform(reg_test)
reg_predictions.select("session_total_spending", "prediction").show(10, truncate=False)


In [ ]:
mae_eval = RegressionEvaluator(labelCol="session_total_spending", predictionCol="prediction", metricName="mae")
mse_eval = RegressionEvaluator(labelCol="session_total_spending", predictionCol="prediction", metricName="mse")
rmse_eval = RegressionEvaluator(labelCol="session_total_spending", predictionCol="prediction", metricName="rmse")
r2_eval = RegressionEvaluator(labelCol="session_total_spending", predictionCol="prediction", metricName="r2")

reg_mae = mae_eval.evaluate(reg_predictions)
reg_mse = mse_eval.evaluate(reg_predictions)
reg_rmse = rmse_eval.evaluate(reg_predictions)
reg_r2 = r2_eval.evaluate(reg_predictions)

print(f"MAE  : {reg_mae:.4f}")
print(f"MSE  : {reg_mse:.4f}")
print(f"RMSE : {reg_rmse:.4f}")
print(f"R2   : {reg_r2:.4f}")

---
## 13. Classification — Spark MLlib

**Target:** `will_purchase` (1 = the session contains a purchase, 0 = browsing-only session).

This is the classic e-commerce conversion-prediction task, and — per Section 14 — every input feature is
derived only from `view`/`cart` behavior, never from the purchase events themselves.

In [ ]:
clf_feature_cols = [
    "session_event_count", "session_view_count", "session_cart_count",
    "session_unique_products", "session_unique_brands", "session_unique_categories",
    "session_avg_price_viewed", "session_max_price_viewed", "session_duration_seconds",
    "session_start_hour", "session_dayofweek",
]

classification_data = session_dataset.na.drop(subset=clf_feature_cols + ["will_purchase"])

print("Class balance:")
classification_data.groupBy("will_purchase").count().show()

clf_train, clf_test = classification_data.randomSplit([0.8, 0.2], seed=42)
clf_train = clf_train.cache()
print("Classification train rows:", clf_train.count(), " | test rows:", clf_test.count())

In [ ]:
try:
    from xgboost.spark import SparkXGBClassifier
except ImportError:
    from sparkxgb import XGBoostClassifier as SparkXGBClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler

# VectorAssembler + Pipeline are used as required by Spark MLlib
clf_assembler = VectorAssembler(inputCols=clf_feature_cols, outputCol="features")

# Spark ML XGBoost Classifier (Spark MLlib Pipeline stage)
try:
    xgb_classifier = SparkXGBClassifier(
        features_col="features", label_col="will_purchase",
        num_workers=2, n_estimators=100, max_depth=8, seed=42
    )
except Exception:
    xgb_classifier = SparkXGBClassifier(
        featuresCol="features", labelCol="will_purchase"
    )

clf_pipeline = Pipeline(stages=[clf_assembler, xgb_classifier])
clf_model = clf_pipeline.fit(clf_train)

clf_predictions = clf_model.transform(clf_test)
clf_predictions.select("will_purchase", "prediction", "probability").show(10, truncate=False)


In [ ]:
acc_eval = MulticlassClassificationEvaluator(labelCol="will_purchase", predictionCol="prediction", metricName="accuracy")
prec_eval = MulticlassClassificationEvaluator(labelCol="will_purchase", predictionCol="prediction", metricName="weightedPrecision")
rec_eval = MulticlassClassificationEvaluator(labelCol="will_purchase", predictionCol="prediction", metricName="weightedRecall")
f1_eval = MulticlassClassificationEvaluator(labelCol="will_purchase", predictionCol="prediction", metricName="f1")
auc_eval = BinaryClassificationEvaluator(labelCol="will_purchase", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

clf_accuracy = acc_eval.evaluate(clf_predictions)
clf_precision = prec_eval.evaluate(clf_predictions)
clf_recall = rec_eval.evaluate(clf_predictions)
clf_f1 = f1_eval.evaluate(clf_predictions)
clf_auc = auc_eval.evaluate(clf_predictions)

print(f"Accuracy  : {clf_accuracy:.4f}")
print(f"Precision : {clf_precision:.4f}")
print(f"Recall    : {clf_recall:.4f}")
print(f"F1        : {clf_f1:.4f}")
print(f"ROC-AUC   : {clf_auc:.4f}")

## 15. Train / Test Split

Both models use a random **80% / 20%** split with `seed=42` (Spark `randomSplit`). A random split is
appropriate here because sessions are treated as i.i.d. units for this analysis (no forecasting of *future*
sessions from *past* sessions is being claimed); the dataset covers a single month, and the goal is
in-sample behavioral prediction rather than out-of-time forecasting. Class distribution for the
classification target (`will_purchase`) was inspected above rather than artificially balanced — per the
requirements, minority-class (purchasing) sessions are **not** deleted to force balance; class weighting or
resampling is left as noted future work (Section 19).

---
## 16. Post-Cleaning / ML Visualization — Exactly 5 Plots

All five plots below use Matplotlib and are built from small Spark aggregations, samples, or the (already
small, session-level) test-set prediction outputs — never a full collect of the 42M-row event table.

### Plot 4 — Engineered Feature Distribution (Session Duration)

In [ ]:
duration_sample_pd = (
    session_dataset.select("session_duration_seconds")
    .filter(F.col("session_duration_seconds").isNotNull() & (F.col("session_duration_seconds") >= 0))
    .sample(withReplacement=False, fraction=0.01, seed=42)
    .toPandas()
)
print("Sample size:", len(duration_sample_pd))

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(duration_sample_pd["session_duration_seconds"] / 60.0, bins=60, color="#4C72B0", edgecolor="white")
ax.set_title("Session Duration Distribution (Sampled)")
ax.set_xlabel("Session Duration (minutes)")
ax.set_ylabel("Frequency")
ax.set_xlim(0, duration_sample_pd["session_duration_seconds"].quantile(0.95) / 60.0)
plt.tight_layout()
plt.savefig("ml_04_session_duration_distribution.png")
plt.show()

### Plot 5 — Feature Relationship: Browsing Intensity vs. Purchase Rate

In [ ]:
# Spark-side aggregation: purchase rate by session_view_count bucket
relationship_pd = (
    session_dataset
    .withColumn(
        "view_bucket",
        F.when(F.col("session_view_count") <= 2, "1-2")
         .when(F.col("session_view_count") <= 5, "3-5")
         .when(F.col("session_view_count") <= 10, "6-10")
         .when(F.col("session_view_count") <= 20, "11-20")
         .otherwise("21+")
    )
    .groupBy("view_bucket")
    .agg(F.avg("will_purchase").alias("purchase_rate"), F.count("*").alias("n_sessions"))
    .toPandas()
)

bucket_order = ["1-2", "3-5", "6-10", "11-20", "21+"]
relationship_pd["view_bucket"] = relationship_pd["view_bucket"].astype("category")
relationship_pd["view_bucket"] = relationship_pd["view_bucket"].cat.set_categories(bucket_order)
relationship_pd = relationship_pd.sort_values("view_bucket")
print(relationship_pd)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(relationship_pd["view_bucket"], relationship_pd["purchase_rate"] * 100, color="#55A868")
ax.set_title("Purchase Rate by Session View-Count Bucket")
ax.set_xlabel("Number of Views in Session")
ax.set_ylabel("Purchase Rate (%)")
plt.tight_layout()
plt.savefig("ml_05_views_vs_purchase_rate.png")
plt.show()

### Plot 6 — Regression: Actual vs. Predicted Spending

In [ ]:
reg_pred_pd = reg_predictions.select("session_total_spending", "prediction").toPandas()

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(reg_pred_pd["session_total_spending"], reg_pred_pd["prediction"], alpha=0.3, s=12, color="#4C72B0")
max_val = max(reg_pred_pd["session_total_spending"].max(), reg_pred_pd["prediction"].max())
ax.plot([0, max_val], [0, max_val], color="red", linestyle="--", label="Perfect prediction")
ax.set_title("Regression: Actual vs. Predicted Session Spending")
ax.set_xlabel("Actual Spending ($)")
ax.set_ylabel("Predicted Spending ($)")
ax.legend()
plt.tight_layout()
plt.savefig("ml_06_regression_actual_vs_predicted.png")
plt.show()

### Plot 7 — Regression Residuals

In [ ]:
reg_pred_pd["residual"] = reg_pred_pd["session_total_spending"] - reg_pred_pd["prediction"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(reg_pred_pd["prediction"], reg_pred_pd["residual"], alpha=0.3, s=12, color="#DD8452")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuals vs. Predicted Spending")
axes[0].set_xlabel("Predicted Spending ($)")
axes[0].set_ylabel("Residual (Actual - Predicted)")

axes[1].hist(reg_pred_pd["residual"], bins=50, color="#DD8452", edgecolor="white")
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual ($)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("ml_07_regression_residuals.png")
plt.show()

### Plot 8 — Classification Confusion Matrix

In [ ]:
import numpy as np

conf_matrix_pd = (
    clf_predictions.groupBy("will_purchase", "prediction")
    .count()
    .toPandas()
)

labels = [0, 1]
matrix = np.zeros((2, 2), dtype=int)
for _, row in conf_matrix_pd.iterrows():
    i = labels.index(int(row["will_purchase"]))
    j = labels.index(int(row["prediction"]))
    matrix[i, j] = int(row["count"])

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(matrix, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["No Purchase", "Purchase"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["No Purchase", "Purchase"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Classification Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{matrix[i, j]:,}", ha="center", va="center",
                color="white" if matrix[i, j] > matrix.max() / 2 else "black", fontsize=12)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("ml_08_classification_confusion_matrix.png")
plt.show()

---
## 18. Model Results Summary

**Regression (predicting `session_total_spending` for purchasing sessions):**

| Metric | Result |
| ------ | -----: |
| MAE    | see `reg_mae` printed above |
| MSE    | see `reg_mse` printed above |
| RMSE   | see `reg_rmse` printed above |
| R²     | see `reg_r2` printed above |

**Classification (predicting `will_purchase`):**

| Metric    | Result |
| --------- | -----: |
| Accuracy  | see `clf_accuracy` printed above |
| Precision | see `clf_precision` printed above |
| Recall    | see `clf_recall` printed above |
| F1        | see `clf_f1` printed above |
| ROC-AUC   | see `clf_auc` printed above |

*(Run the notebook top-to-bottom against the live HDFS/Spark cluster to populate the actual numeric values —
this notebook was authored/extended outside that environment, so exact figures depend on the real cluster
run.)*

In [ ]:
print("=== REGRESSION RESULTS ===")
print(f"MAE  : {reg_mae:.4f}")
print(f"MSE  : {reg_mse:.4f}")
print(f"RMSE : {reg_rmse:.4f}")
print(f"R2   : {reg_r2:.4f}")

print("\n=== CLASSIFICATION RESULTS ===")
print(f"Accuracy  : {clf_accuracy:.4f}")
print(f"Precision : {clf_precision:.4f}")
print(f"Recall    : {clf_recall:.4f}")
print(f"F1        : {clf_f1:.4f}")
print(f"ROC-AUC   : {clf_auc:.4f}")

---
## 19. Final Conclusions

### EDA Findings
The event log follows a classic e-commerce funnel: overwhelmingly `view` events, fewer `cart` events, and
the fewest `purchase` events, meaning conversion is a rare, high-value outcome — this shaped both the
regression target (restricted to converting sessions) and the expectation that the classification target
is imbalanced.

### Data Quality
`category_code` and `brand` had substantial missingness, handled via explicit `"Unknown"` fills rather than
row deletion; the 2 missing `user_session` values were filled with a placeholder for the same reason.
`event_time` was converted from string to a proper Spark `timestamp`. Duplicate exact-row events were
counted but not blindly deleted. `price` outliers (via IQR) were investigated and left in place, since they
correspond to legitimate premium products rather than errors, consistent with the strong right-skew seen
in the EDA price histogram.

### Feature Engineering
Session-level features (event/view/cart counts, unique products/brands/categories, average/max price
viewed, session duration, start hour, day of week) were engineered entirely from `view` and `cart` events,
deliberately excluding anything derived from `purchase` events to keep the feature set leakage-safe for
both downstream models.

### Regression
- **Target:** `session_total_spending`, computed only for sessions that purchased.
- **Model:** `SparkXGBRegressor (Spark ML XGBoost)` (Spark MLlib) on an 80/20 split, seed 42.
- **Metrics:** MAE / MSE / RMSE / R² as printed above.
- **Limitations:** Restricting to purchasing sessions means the model does not itself predict *whether* a
  session converts (that's the classifier's job) — the two models are complementary, not a single funnel
  model. Spending amounts can be heavy-tailed (a few very large purchases), which inflates RMSE relative to
  MAE; a log-transformed target could be explored in future work.

### Classification
- **Target:** `will_purchase` — whether a session contains at least one purchase.
- **Model:** `SparkXGBClassifier (Spark ML XGBoost)` (Spark MLlib) on an 80/20 split, seed 42.
- **Metrics:** Accuracy / Precision / Recall / F1 / ROC-AUC as printed above, and a confusion matrix (Plot 8).
- **Limitations:** Purchasing sessions are the minority class (per the EDA funnel), so accuracy alone can be
  misleading — Recall/F1/ROC-AUC and the confusion matrix are the more informative metrics for this
  imbalance, and no minority-class rows were deleted to artificially balance the classes.

### Overall Conclusion
Pre-purchase browsing signals (breadth of products/brands/categories viewed, cart activity, session
duration and timing, and price sensitivity) carry real predictive signal for both *whether* a session
converts and, conditional on converting, *how much* it's likely to spend — while every feature used was
constructed to be available strictly before the outcome is known, avoiding leakage from the purchase event
itself into either model.

### Future Work
- Hyperparameter tuning (`numTrees`, `maxDepth`, `maxBins`) via `CrossValidator` / `TrainValidationSplit`.
- k-fold cross-validation for more robust metric estimates than a single train/test split.
- Additional behavioral features: repeat-visit history per `user_id` across sessions, cart-abandonment
  signals, category-affinity features.
- Alternative Spark MLlib models: `GBTRegressor`/`GBTClassifier`, `LogisticRegression` with regularization,
  `LinearRegression` as a simpler regression baseline.
- Time-aware validation (e.g. train on the first three weeks of October, validate on the last week) if the
  goal shifts from in-sample behavioral prediction to genuine forecasting.
- More advanced session features such as scroll/dwell-time proxies (if available in extended datasets),
  device/channel information, and price-drop or promotion flags.